In [0]:
%spark.pyspark
from pyspark.sql import functions as F
from pyspark.sql.window import Window

users = spark.createDataFrame([
    ("u1", "Berlin"),
    ("u2", "Berlin"),
    ("u3", "Munich"),
    ("u4", "Hamburg")
], ["user_id", "city"])

orders = spark.createDataFrame([
    ("o1", "u1", "p1", 2, 10.0),
    ("o2", "u1", "p2", 1, 30.0),
    ("o3", "u2", "p1", 1, 10.0),
    ("o4", "u2", "p3", 5, 7.0),
    ("o5", "u3", "p2", 3, 30.0),
    ("o6", "u3", "p3", 1, 7.0),
    ("o7", "u4", "p1", 10, 10.0)
], ["order_id", "user_id", "product_id", "qty", "price"])

products = spark.createDataFrame([
    ("p1", "Ring VOLA"),
    ("p2", "Ring POROG"),
    ("p3", "Ring TISHINA")
], ["product_id", "product_name"])

users.show()
orders.show()
products.show()

In [1]:
%spark.pyspark

from pyspark.sql import functions as F

orders_with_revenue = orders.withColumn(
    "revenue", 
    F.col("qty").cast("double") * F.col("price").cast("double")
)

enriched_df = orders_with_revenue \
    .join(users, "user_id", "inner") \
    .join(products, "product_id", "inner")

mart_aggregated = enriched_df.groupBy("city", "product_id", "product_name") \
    .agg(
        F.count("order_id").alias("orders_cnt"),
        F.sum("qty").alias("qty_sum"),
        F.sum("revenue").alias("revenue_sum")
    )

mart_aggregated.show()

In [2]:
%spark.pyspark
from pyspark.sql.window import Window

window_spec = Window.partitionBy("city").orderBy(F.desc("revenue_sum"))

mart_with_rank = mart_aggregated.withColumn("rank", F.row_number().over(window_spec))

mart_city_top_products = mart_with_rank.filter(F.col("rank") <= 2).drop("rank")

mart_city_top_products.show()

In [3]:
%spark.pyspark

path = "/tmp/sandbox_zeppelin/mart_city_top_products/"
mart_city_top_products.write.mode("overwrite").parquet(path)

final_check = spark.read.parquet(path)
print("Результат успешно прочитан из Parquet:")
final_check.show()